<a href="https://colab.research.google.com/github/Emmanuel-Yerbo/GIS/blob/main/Python%20for%20Urban%20Analysis/Chapter-5/Chap_5_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Chapter 5 — The Urban Profile: Base Maps & Composed Layers
## Python for Urban Analysis — Ghana Edition

---


### Learning Objectives
1. Create polished base maps with contextily
2. Compose multiple urban layers (roads, buildings, POIs, green space) into a single map
3. Build a reusable multi-panel layout showing all city layers
4. Apply professional cartographic styling

## 5.0 — Install & Configure

In [ ]:
!pip install -q osmnx geopandas contextily matplotlib mapclassify

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import osmnx as ox
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import contextily as cx
from pathlib import Path

# ── Configuration ─────────────────────────────────────────────
STUDY_AREA       = 'Accra, Ghana'
CRS_WEB_MERCATOR = 'EPSG:3857'
CRS_UTM_ACCRA    = 'EPSG:32630'
DARK_BG          = '#0e1117'
ACCENT_CMAP      = 'YlOrRd'
AMENITY_COLOR    = '#00e5ff'

AMENITY_TAGS = {
    'amenity': [
        'hospital', 'clinic', 'pharmacy',
        'school', 'university', 'library',
        'marketplace', 'bank', 'atm',
        'police', 'fire_station',
        'place_of_worship', 'restaurant', 'fuel',
    ]
}
BUILDING_TAGS = {'building': True}
LEISURE_TAGS = {
    'leisure': ['park', 'playground', 'sports_centre', 'garden', 'stadium']
}

DATA_DIR    = Path('data')
FIGURES_DIR = Path('outputs') / 'figures'
for d in [DATA_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Study area: {STUDY_AREA}')


## 5.1 — Pull Layers Directly from OpenStreetMap

Each layer is fetched directly in this notebook so Chapter 5 runs independently.

In [ ]:
print(f' Fetching Urban Layers from OpenStreetMap for: {STUDY_AREA} ...')

# 1. Boundary
boundary = ox.geocode_to_gdf(STUDY_AREA)
boundary_wm = boundary.to_crs(CRS_WEB_MERCATOR)
print(f'   ✓ Boundary   : Geocoded {STUDY_AREA}')

# 2. Buildings
buildings_raw = ox.features_from_place(STUDY_AREA, tags=BUILDING_TAGS)
buildings = buildings_raw[
    buildings_raw.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])
].copy()
buildings['area_m2'] = buildings.to_crs(CRS_UTM_ACCRA).geometry.area.round(1)
buildings_wm = buildings.to_crs(CRS_WEB_MERCATOR)
print(f'   ✓ Buildings  : {len(buildings):,} footprints (area computed)')

# 3. Amenities
amenities_raw = ox.features_from_place(STUDY_AREA, tags=AMENITY_TAGS).copy()
amenities = amenities_raw.copy()
amenities['geometry'] = amenities.geometry.centroid
amenities = amenities[amenities.geometry.within(boundary.geometry.iloc[0])].copy()
amenities_wm = amenities.to_crs(CRS_WEB_MERCATOR)
print(f'   ✓ Amenities  : {len(amenities):,} POIs')

# 4. Road Network
G_drive = ox.graph_from_place(STUDY_AREA, network_type='drive')
roads = ox.graph_to_gdfs(G_drive, nodes=False, edges=True)
roads_wm = roads.to_crs(CRS_WEB_MERCATOR)
print(f'   ✓ Roads      : {len(roads):,} edges')

# 5. Leisure / Green Space
leisure_raw = ox.features_from_place(STUDY_AREA, tags=LEISURE_TAGS)
leisure = leisure_raw[
    leisure_raw.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])
].copy()
leisure_wm = leisure.to_crs(CRS_WEB_MERCATOR)
print(f'   ✓ Leisure    : {len(leisure):,} features')


## 5.2 — Construct the 2×2 Multi-Panel City Dashboard

We compile the maps into a 2×2 grid dashboard showcasing different dimensions of the city.

In [ ]:
def make_layer_panel(ax, title, plot_fn):
    ax.set_facecolor(DARK_BG)
    plot_fn(ax)
    boundary_wm.plot(ax=ax, facecolor='none', edgecolor='white', linewidth=1.5)
    cx.add_basemap(ax, source=cx.providers.CartoDB.DarkMatter, zoom=13)
    ax.set_axis_off()
    ax.set_title(title, fontsize=11, fontweight='bold', color='white', pad=8)

print('Composing the Urban Profile (2×2 panels) ...')
fig, axes = plt.subplots(2, 2, figsize=(20, 20))
fig.patch.set_facecolor(DARK_BG)
fig.suptitle(f'URBAN PROFILE — {STUDY_AREA.upper()}', fontsize=22, fontweight='bold', color='white', y=0.98)

# Panel 1: Buildings
def plot_buildings(ax):
    q99 = buildings_wm['area_m2'].quantile(0.99)
    bldg = buildings_wm[buildings_wm['area_m2'] <= q99]
    bldg.plot(
        ax=ax, column='area_m2', cmap=ACCENT_CMAP,
        scheme='quantiles', k=7, linewidth=0.05, edgecolor='#222', alpha=0.9
    )
make_layer_panel(axes[0, 0], 'Buildings (by footprint area)', plot_buildings)

# Panel 2: Roads
def plot_roads(ax):
    roads_wm.plot(ax=ax, color='#ff9100', linewidth=0.3, alpha=0.7)
make_layer_panel(axes[0, 1], 'Road Network (drivable)', plot_roads)

# Panel 3: POIs
def plot_amenities(ax):
    amenities_wm.plot(ax=ax, color=AMENITY_COLOR, markersize=5, alpha=0.7, zorder=5)
make_layer_panel(axes[1, 0], 'Points of Interest (amenities)', plot_amenities)

# Panel 4: Combined Composition
def plot_all(ax):
    if len(leisure_wm) > 0:
        leisure_wm.plot(ax=ax, color='#66bb6a', alpha=0.4)
    buildings_wm.plot(ax=ax, color='#ffc107', linewidth=0, alpha=0.5)
    roads_wm.plot(ax=ax, color='#78909c', linewidth=0.2, alpha=0.5)
    amenities_wm.plot(ax=ax, color=AMENITY_COLOR, markersize=3, alpha=0.8, zorder=5)
make_layer_panel(axes[1, 1], 'All Layers Composed', plot_all)

legend_items = [
    mpatches.Patch(color='#ffc107', label='Buildings'),
    mpatches.Patch(color='#78909c', label='Roads'),
    mpatches.Patch(color=AMENITY_COLOR, label='Amenities'),
    mpatches.Patch(color='#66bb6a', label='Green/Leisure'),
]
axes[1, 1].legend(
    handles=legend_items, loc='lower right', fontsize=8,
    facecolor='#1a1a2e', edgecolor='white', labelcolor='white', framealpha=0.9
)

fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(FIGURES_DIR / 'ch5_urban_profile.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()


## 5.3 — Build the Combined Single-Canvas Hero Map

This overlays all layers onto a single beautiful canvas using proper layering and alpha transparency to avoid visual overload.

In [ ]:
print('Creating hero single-canvas map ...')

fig, ax = plt.subplots(1, 1, figsize=(18, 18))
fig.patch.set_facecolor(DARK_BG)
ax.set_facecolor(DARK_BG)

# Layer 1: Leisure/green spaces
if len(leisure_wm) > 0:
    leisure_wm.plot(ax=ax, color='#2e7d32', alpha=0.5, zorder=1)

# Layer 2: Buildings by Area
q99 = buildings_wm['area_m2'].quantile(0.99)
bldg = buildings_wm[buildings_wm['area_m2'] <= q99]
bldg.plot(
    ax=ax, column='area_m2', cmap=ACCENT_CMAP,
    scheme='quantiles', k=7, linewidth=0.05, edgecolor='#111', alpha=0.85,
    legend=False, zorder=2
)

# Layer 3: Roads
roads_wm.plot(ax=ax, color='#546e7a', linewidth=0.2, alpha=0.3, zorder=3)

# Layer 4: Amenities
amenities_wm.plot(ax=ax, color=AMENITY_COLOR, markersize=6, alpha=0.9, zorder=5)

# Boundary outline
boundary_wm.plot(ax=ax, facecolor='none', edgecolor='white', linewidth=3, zorder=10)

cx.add_basemap(ax, source=cx.providers.CartoDB.DarkMatter, zoom=14)
ax.set_axis_off()
ax.set_title(f'Complete Urban Profile — {STUDY_AREA}', fontsize=20, fontweight='bold', color='white', pad=20)

fig.tight_layout()
fig.savefig(FIGURES_DIR / 'ch5_hero_map.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()


## ✅ Chapter 5 Complete

### What you accomplished:
- Loaded all data layers together.
- Converted static layout panels into a 2×2 city dashboard.
- Styled and saved a stunning multi-layer urban profile map.

### Next → Chapter 6: Scaling to the Livability Index across Districts!